In [41]:
import torch
import torch.nn as nn
from torch.optim import AdamW
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torch.optim.lr_scheduler import OneCycleLR
from src.model.architecture.madnet.madnet import MADNet
from src.data.dataset import AudioDataset
import pandas as pd
import matplotlib.pyplot as plt
import librosa
from pathlib import Path
from tqdm import tqdm
import numpy as np

In [58]:
train_path = r"D:\Project\AI Generated Audio Detection\datasets\raw\for-norm\for-norm\training"
val_path = r"D:\Project\AI Generated Audio Detection\datasets\raw\for-norm\for-norm\validation"
test_path = r"D:\Project\AI Generated Audio Detection\datasets\raw\for-norm\for-norm\testing"

train_save_path = r"D:\Project\AI Generated Audio Detection\datasets\processed\training\train.pt"
val_save_path = r"D:\Project\AI Generated Audio Detection\datasets\processed\validation\val.pt"
test_save_path = r"D:\Project\AI Generated Audio Detection\datasets\processed\testing\test.pt"

In [42]:
def save_preprocessed(root_path, output_path, sr=16000, target_length=48000, n_mels=224):
    root = Path(root_path)
    specs, labels = [], []

    for label_idx, label_name in enumerate(["fake", "real"]):  # fake=0, real=1
        input_folder = root / label_name
        for file_path in tqdm(list(input_folder.iterdir()), desc=label_name):
            if not file_path.is_file():
                continue
            try:
                wav, _ = librosa.load(str(file_path), sr=sr, mono=True)
                wav = wav / (np.max(np.abs(wav)) + 1e-6)

                if len(wav) < target_length:
                    pad_total = target_length - len(wav)
                    wav = np.pad(wav, (pad_total // 2, pad_total - pad_total // 2))
                else:
                    start = (len(wav) - target_length) // 2
                    wav = wav[start:start + target_length]

                spec = librosa.feature.melspectrogram(y=wav, sr=sr, n_fft=1024, hop_length=512, n_mels=n_mels)
                spec = librosa.power_to_db(spec, ref=np.max, top_db=80.0)
                spec = (spec + 80) / 80

                spec = torch.tensor(spec, dtype=torch.float32).unsqueeze(0).unsqueeze(0)  # (1, 1, H, W)
                spec = F.interpolate(spec, size=(224, 224), mode='bilinear', align_corners=False)
                spec = spec.squeeze(0)  # (1, 224, 224)

                specs.append(spec)  # (1, H, W)
                labels.append(label_idx)

            except Exception as e:
                print(f"Skipping {file_path}: {e}")

    torch.save({"specs": torch.stack(specs), "labels": torch.tensor(labels, dtype=torch.long)}, output_path)
    print(f"Saved {len(specs)} samples to {output_path}")

In [43]:
save_preprocessed(train_path, train_save_path)
save_preprocessed(val_path, val_save_path)
save_preprocessed(test_path, test_save_path)

real:   4%|▎         | 949/26941 [00:06<02:54, 149.24it/s]

Skipping D:\Project\AI Generated Audio Detection\datasets\raw\for-norm\for-norm\training\real\file11064.wav_16k.wav_norm.wav_mono.wav_silence.wav: zero-size array to reduction operation maximum which has no identity


real:  17%|█▋        | 4713/26941 [00:32<02:18, 159.97it/s]

Skipping D:\Project\AI Generated Audio Detection\datasets\raw\for-norm\for-norm\training\real\file15440.wav_16k.wav_norm.wav_mono.wav_silence.wav: zero-size array to reduction operation maximum which has no identity


real: 100%|██████████| 26941/26941 [03:11<00:00, 140.54it/s]


Saved 53866 samples to D:\Project\AI Generated Audio Detection\datasets\processed\training\train.pt


real: 100%|██████████| 5400/5400 [00:41<00:00, 129.46it/s]


Saved 10798 samples to D:\Project\AI Generated Audio Detection\datasets\processed\validation\val.pt


real: 100%|██████████| 2264/2264 [00:15<00:00, 146.30it/s]


Saved 4634 samples to D:\Project\AI Generated Audio Detection\datasets\processed\training\test.pt


In [45]:
class SpectrogramDataset(Dataset):
    def __init__(self, pt_path, transform=None):
        data = torch.load(pt_path)
        self.specs  = data["specs"]   # (N, 1, H, W)
        self.labels = data["labels"]  # (N,)
        self.transform = transform

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        spec = self.specs[idx]
        if self.transform:
            spec = self.transform(spec)
        return spec, self.labels[idx]

In [ ]:
train_loader = DataLoader(SpectrogramDataset(train_save_path), batch_size=32, shuffle=True)
val_loader = DataLoader(SpectrogramDataset(val_save_path), batch_size=32, shuffle=False)
test_loader = DataLoader(SpectrogramDataset(test_save_path), batch_size=32, shuffle=False)

C:\Users\VUONG\AppData\Local\Temp\ipykernel_13252\333377070.py:3: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  data = torch.load(pt_path)


In [ ]:
class Trainer:
    def __init__(self, model: MADNet, train_path: str, val_path: str, num_classes: int=2,
                 batch_size: int=32, lr: float=1e-3, weight_decay: float=1e-5, epochs: int=100):
        self.device = 'cuda' if torch.cuda.is_available() else 'cpu'

        self.model = model.to(self.device)

        if self.device == 'cuda' and torch.cuda.device_count() > 1:
            print(f"Using DataParallel {torch.cuda.device_count()} GPUs")
            self.model = nn.DataParallel(self.model)

        self.train_loader = DataLoader(AudioDataset(train_path), batch_size=batch_size, shuffle=True, num_workers=4, pin_memory=True)
        self.val_loader = DataLoader(AudioDataset(val_path), batch_size=batch_size, shuffle=False, num_workers=4, pin_memory=True)
        self.epochs = epochs
        
        self.optimizer = AdamW(self.model.parameters(), lr=lr, weight_decay=weight_decay)
        self.criterion = nn.BCEWithLogitsLoss()
        self.scheduler = OneCycleLR(self.optimizer, max_lr=lr*10, epochs=epochs, steps_per_epoch=len(self.train_loader), pct_start=0.3, anneal_strategy='cos', div_factor=25.0, final_div_factor=1e4)


    def _train_epoch(self, loader, train):
        self.model.train() if train else self.model.eval()
        context = torch.enable_grad() if train else torch.no_grad()

        total_loss = correct = total = 0.0

        with context:
            for imgs, labels in loader:
                imgs, labels = imgs.to(self.device), labels.to(self.device).float().unsqueeze(1)

                if train:
                    self.optimizer.zero_grad()

                out = self.model(imgs)
                loss = self.criterion(out, labels)

                if train:
                    loss.backward()
                    self.optimizer.step()
                    self.scheduler.step()

                total_loss += loss.item() * imgs.size(0)

                preds = (torch.sigmoid(out) > 0.5).float()
                correct += (preds == labels).sum().item()
                total += labels.numel()
        return total_loss / total, correct / total
    

    def train(self, save_path: str='best_model.pt', history_path: str='history.csv'):
        best_val_loss = float('inf')
        history = {'epoch': [], 'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
        header = f"{'Epoch':>6} | {'Train Loss':>10} | {'Train Acc':>9} | {'Val Loss':>8} | {'Val Acc':>8} | {'':>18}"
        divider = '-' * len(header)
        print(f"\nTraining for up to {self.epochs} epochs on {self.device}")
        print(divider)
        print(header)
        print(divider)

        for epoch in range(1, self.epochs + 1):
            train_loss, train_acc = self._train_epoch(self.train_loader, train=True)
            val_loss, val_acc = self._train_epoch(self.val_loader, train=False)

            history['epoch'].append(epoch)
            history['train_loss'].append(train_loss)
            history['train_acc'].append(train_acc)
            history['val_loss'].append(val_loss)
            history['val_acc'].append(val_acc)

            print(f"{epoch:>6} | {train_loss:>10.4f} | {train_acc:>8.2%} | {val_loss:>8.4f} | {val_acc:>8.2%}")

            if val_loss < best_val_loss:
                best_val_loss = val_loss
                core = self.model.module if isinstance(self.model, nn.DataParallel) else self.model
                torch.save(core.state_dict(), save_path)

        self.history_df = pd.DataFrame(history)
        self.history_df.to_csv(history_path, index=False)
        print(f"History saved to '{history_path}'\n")